# Session 12 — ZeRO-0/1/2/3 on 32 Virtual GPUs
**Hari — ERA V5 Assignment 12**

32 logical ranks on a single CPU. These are not physical GPUs. We validate shard coverage and Adam-update equivalence using a real PyTorch MLP, then estimate mixed-precision persistent model-state memory and collective payload. No real distributed timing is claimed.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
R=32
torch.manual_seed(12)
np.random.seed(12)
model=torch.nn.Sequential(torch.nn.Linear(16,64),torch.nn.Tanh(),torch.nn.Linear(64,4))
params=list(model.parameters())
w=torch.cat([p.detach().flatten() for p in params]); P=w.numel()
G=[]; losses=[]
for r in range(R):
    x=torch.randn(2,16); y=torch.randn(2,4)
    model.zero_grad(set_to_none=True)
    loss=torch.nn.functional.mse_loss(model(x),y)
    loss.backward()
    losses.append(loss.item())
    G.append(torch.cat([p.grad.detach().flatten().clone() for p in params]))
G=torch.stack(G); g=G.mean(0)
print('device=CPU, virtual ranks=',R,'P=',P,'gradient shape=',tuple(G.shape),'mean loss=',np.mean(losses))

## Shard and reconstruct
Each shard owns a disjoint range. ZeRO-0 replicates all training state. ZeRO-1 shards optimizer state. ZeRO-2 shards optimizer state and gradients. ZeRO-3 also shards weights. Stage 3 temporarily gathers weights around forward/backward computation.

In [ ]:
S=[slice(P*r//R,P*(r+1)//R) for r in range(R)]
assert sorted(i for s in S for i in range(s.start,s.stop))==list(range(P))
assert torch.equal(torch.cat([w[s] for s in S]),w)
assert torch.equal(torch.cat([g[s] for s in S]),g)
print('Coverage and reconstruction PASS; min/max shard sizes:',min(s.stop-s.start for s in S), max(s.stop-s.start for s in S))

## Exact Adam-update decomposition test
We compare an unsharded Adam update with rank-owned slices. Stage 2/3 gradient slices are obtained via a simulated reduce-scatter (sum rank gradients on each slice, divide by 32). This is not an implementation of network collectives. Nonzero moment states and a time index of 5 exercise bias correction.

In [ ]:
beta1,beta2,eps,lr,t=.9,.999,1e-8,1e-3,5
m=torch.linspace(-.015,.015,P); v=torch.linspace(.001,.003,P)
def adam(w,g,m,v):
    m1=beta1*m+(1-beta1)*g; v1=beta2*v+(1-beta2)*g*g
    return w-lr*(m1/(1-beta1**t))/(torch.sqrt(v1/(1-beta2**t))+eps)
ref=adam(w,g,m,v)
for stage in range(4):
    parts=[]
    for s in S:
        gs=G[:,s].sum(0)/R if stage>=2 else g[s]
        parts.append(adam(w[s],gs,m[s],v[s]))
    out=torch.cat(parts); error=(out-ref).abs().max().item()
    assert torch.allclose(out,ref,atol=2e-7,rtol=1e-6)
    print(f'ZeRO-{stage}: max difference {error:.3e} — PASS')

## Persistent model-state memory
Conceptual mixed precision: FP16 weights 2 B/parameter, FP16 gradients 2 B/parameter, FP32 master copy 4 B/parameter, Adam FP32 first and second moments 8 B/parameter. The demonstration update above uses FP32 tensors; numbers here are **not measured CPU/CUDA allocations**. We exclude activations, buffers, fragmentation and runtime overhead.

In [ ]:
def mem(n,stage):
    return {'weights':2*n/(R if stage>=3 else 1),
            'gradients':2*n/(R if stage>=2 else 1),
            'optimizer+master':12*n/(R if stage>=1 else 1)}
for st in range(4):
    print(f'ZeRO-{st}: {sum(mem(P,st).values())/1024:.2f} KiB/rank; 1B-parameter case: {sum(mem(10**9,st).values())/2**30:.3f} GiB/rank')
bottom=np.zeros(4)
fig,ax=plt.subplots(figsize=(8,4))
for k in ('weights','gradients','optimizer+master'):
    val=np.array([mem(P,s)[k]/1024 for s in range(4)])
    ax.bar([f'ZeRO-{s}' for s in range(4)],val,bottom=bottom,label=k);bottom+=val
ax.set_title('32 virtual ranks: estimated persistent memory');ax.set_ylabel('KiB per rank');ax.legend();plt.tight_layout();plt.show()

## Computation versus communication
Ordinary data-parallel forward/backward work remains per rank in each mode. Optimizer updates are partitioned across 32 ranks for stages 1–3 (each rank updates ~1/32 of the parameters). Ring collective idealization: all-reduce = `2(R-1)/R * tensor bytes`, reduce-scatter or all-gather = `(R-1)/R * tensor bytes` per rank. This is payload accounting, **not time**. ZeRO-3 estimates two parameter all-gathers (forward and backward) while retaining sharded storage between iterations; real implementations schedule communication differently.

In [ ]:
def comm(n,stage):
    f=(R-1)/R; pb=2*n;gb=2*n
    if stage==0: return {'gradient reduction':2*f*gb,'parameter gathers':0}
    if stage==1: return {'gradient reduction':2*f*gb,'parameter gathers':f*pb}
    if stage==2: return {'gradient reduction':f*gb,'parameter gathers':f*pb}
    return {'gradient reduction':f*gb,'parameter gathers':2*f*pb}
for st in range(4):
    print(f'ZeRO-{st}: {sum(comm(P,st).values())/1024:.2f} KiB/rank/step payload estimate')
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].bar([f'ZeRO-{s}' for s in range(4)],[sum(comm(P,s).values())/1024 for s in range(4)])
ax[0].set_title('Illustrative communication');ax[0].set_ylabel('KiB per rank/step')
ax[1].bar([f'ZeRO-{s}' for s in range(4)],[1 if s==0 else 1/R for s in range(4)])
ax[1].set_title('Local Adam update fraction');ax[1].set_ylabel('Fraction of total parameters')
plt.tight_layout();plt.show()

## My conclusion
ZeRO solves model-state replication at progressively higher stages. ZeRO-1 shards optimizer and master states, ZeRO-2 also shards persistent gradients, and ZeRO-3 also shards persistent parameters. Under the stated 16-byte mixed-precision state budget, per-rank state for a 1B-parameter model falls from ~14.90 GiB (baseline) to ~0.47 GiB (Stage 3) at 32 ranks, ignoring temporary memory. Computation is not necessarily faster: Stage 3 needs parameter gathers and real performance depends on network throughput, collective scheduling and activation memory. The actual correctness checks here prove partition/reconstruction and Adam update equivalence, not 32-device distributed speedup.